Install Requirements

In [ ]:
%pip install -Uq "unstructured[all-docs]" lxml
%pip install -Uq chromadb tiktoken
%pip install -Uq langchain langchain-community langchain-openai langchain-groq
%pip install -Uq python_dotenv
%pip install "Pillow<12.0.0"

!apt-get install -y poppler-utils

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


Environment Setup

OPENAI_API_KEY

GROQ_API_KEY

In [ ]:
import os, uuid, base64, traceback
from unstructured.partition.pdf import partition_pdf
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr
from IPython.display import Image, display

In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

print("Keys loaded successfully")

Keys loaded successfully


In [ ]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")

Initialize Models & Global Variables

In [ ]:
# Groq LLM for summarization
groq_llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2, api_key=GROQ_API_KEY)

# OpenAI LLM for QA + image descriptions
qa_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, api_key=OPENAI_API_KEY)

# Embeddings for vector store
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# Global retriever for RAG
retriever_global = None

PDF Processing Function

In [ ]:
def process_pdf(file):
    global retriever_global
    try:
        print("🔄 Extracting PDF content...")

        chunks = partition_pdf(
            filename=file,
            strategy="fast",  # Use 'hi_res' if tables/images are needed
            infer_table_structure=True,
            extract_image_block_types=["Image", "Table"],
            extract_image_block_to_payload=True,
            chunking_strategy="by_title",
            max_characters=10000,
            combine_text_under_n_chars=2000,
            new_after_n_chars=6000
        )
        print(f"PDF split into {len(chunks)} chunks")

        # Separate elements
        texts, tables, images_b64 = [], [], []

        for chunk in chunks:
            if "CompositeElement" in str(type(chunk)):
                texts.append(chunk)
                for el in chunk.metadata.orig_elements:
                    if "Table" in str(type(el)):
                        tables.append(el)
                    if "Image" in str(type(el)):
                        images_b64.append(el.metadata.image_base64)

        print(f"Text blocks: {len(texts)} | Tables: {len(tables)} | Images: {len(images_b64)}")

        # Summarize texts & tables with Groq
        prompt_text = """
        Summarize the following content concisely.
        Content: {element}
        Only return the summary.
        """
        prompt = ChatPromptTemplate.from_template(prompt_text)
        chain = prompt | groq_llm | StrOutputParser()

        text_summaries = chain.batch([t.text for t in texts])
        tables_html = [t.metadata.text_as_html for t in tables]
        table_summaries = chain.batch(tables_html) if tables_html else []

        # Summarize images with OpenAI
        image_summaries = []
        for img in images_b64:
            messages = [
                (
                    "user",
                    [
                        {"type": "text", "text": "Describe this image from a research paper."},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img}"}}
                    ],
                )
            ]
            summary = qa_llm.invoke(messages)
            image_summaries.append(summary.content)

        # Create Document objects
        docs = []
        for s, t in zip(text_summaries, texts):
            docs.append(Document(page_content=s, metadata={"id": str(uuid.uuid4()), "modality": "text", "original": t.text}))
        for s, t_html in zip(table_summaries, tables_html):
            docs.append(Document(page_content=s, metadata={"id": str(uuid.uuid4()), "modality": "table", "original": t_html}))
        for img_b64, s in zip(images_b64, image_summaries):
            docs.append(Document(page_content=s, metadata={"id": str(uuid.uuid4()), "modality": "image", "image_b64": img_b64}))

        # Create Chroma vector store
        vectorstore = Chroma.from_documents(docs, embeddings, collection_name="multimodal_rag")
        retriever_global = vectorstore.as_retriever(search_kwargs={"k": 6})

        return "✅ PDF processed successfully! You can now ask questions."

    except Exception as e:
        return f"❌ ERROR processing PDF:\n{traceback.format_exc()}"

RAG QA Function

In [ ]:
def ask_question(question):
    global retriever_global
    if retriever_global is None:
        return "⚠️ Please upload and process a PDF first.", ""

    retrieved_docs = retriever_global.get_relevant_documents(question)

    # Format context for LLM
    context_lines = []
    for i, d in enumerate(retrieved_docs, 1):
        m = d.metadata.get("modality", "text")
        context_lines.append(f"[{i}] {m.upper()} — {d.page_content}")

    context = "\n\n".join(context_lines)

    # Ask LLM (GPT-4o-mini) using context
    prompt_template = f"""
    You answer strictly using the provided context.
    Cite sources using [number].
    If information is missing, say you don't know.

    Question: {question}

    Context:
    {context}
    """

    answer = qa_llm.invoke(prompt_template).content
    citations = "\n".join([f"[{i}] {d.metadata['modality'].upper()}" for i, d in enumerate(retrieved_docs, 1)])
    return answer, citations

Gradio Interface

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 📄 Multimodal PDF RAG Assistant")
    gr.Markdown("Upload a PDF and ask questions with citation grounding.")

    with gr.Row():
        pdf_input = gr.File(file_types=[".pdf"], label="Upload PDF")
        process_btn = gr.Button("Process PDF")
    status_output = gr.Textbox(label="Status")

    question_input = gr.Textbox(label="Ask a question")
    ask_btn = gr.Button("Get Answer")
    answer_output = gr.Textbox(label="Answer", lines=8)
    citation_output = gr.Textbox(label="Cited Sources")

    process_btn.click(process_pdf, inputs=pdf_input, outputs=status_output)
    ask_btn.click(ask_question, inputs=question_input, outputs=[answer_output, citation_output])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f43c75aa8b935217cc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
